In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import json
import base64
import webbrowser
from pathlib import Path

FILE_NC = "grace.nc"                    
OUTPUT_HTML = "dashboard_grace_temporal_nc.html"
VARIABLE_NAME = None                    
STEP = 4                                

nc_path = Path(FILE_NC)
if not nc_path.exists():
    raise FileNotFoundError(
        f"File {FILE_NC!r} tidak ditemukan. Simpan file .py dan .nc dalam folder yang sama."
    )

ds = xr.open_dataset(nc_path)
print("Variabel tersedia:", list(ds.data_vars))
print("Koordinat tersedia:", list(ds.coords))
print(ds)

def find_name(candidates, available):
    lookup = {str(v).lower(): v for v in available}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    for v in available:
        if any(c.lower() in str(v).lower() for c in candidates):
            return v
    return None

available = list(ds.coords) + [d for d in ds.dims if d not in ds.coords]
time_name = find_name(["time", "date", "datetime", "valid_time"], available)
lat_name = find_name(["lat", "latitude"], available)
lon_name = find_name(["lon", "longitude"], available)

if not all([time_name, lat_name, lon_name]):
    ds.close()
    raise ValueError(
        f"Koordinat waktu/lintang/bujur tidak terdeteksi. Nama yang tersedia: {available}"
    )

if VARIABLE_NAME:
    if VARIABLE_NAME not in ds.data_vars:
        ds.close()
        raise KeyError(f"Variabel {VARIABLE_NAME!r} tidak ada. Pilihan: {list(ds.data_vars)}")
    var_name = VARIABLE_NAME
else:
    candidates = [
        name for name, var in ds.data_vars.items()
        if time_name in var.dims and lat_name in var.dims and lon_name in var.dims
    ]
    if not candidates:
        ds.close()
        raise ValueError(
            "Tidak ada variabel dengan dimensi waktu, lintang, dan bujur. "
            "Periksa struktur file dan isi VARIABLE_NAME secara manual."
        )
    var_name = candidates[0]

da = ds[var_name].transpose(time_name, lat_name, lon_name)
da = da.isel({lat_name: slice(None, None, STEP), lon_name: slice(None, None, STEP)})

lat = np.asarray(da[lat_name].values, dtype=np.float32)
lon = np.asarray(da[lon_name].values, dtype=np.float32)

raw_time = da[time_name].values
time_coord = ds[time_name]
units = (
    time_coord.attrs.get("units")
    or time_coord.encoding.get("units")
    or ""
)
calendar = (
    time_coord.attrs.get("calendar")
    or time_coord.encoding.get("calendar")
    or "standard"
)

if np.issubdtype(np.asarray(raw_time).dtype, np.datetime64):
    dates = pd.to_datetime(raw_time, errors="coerce", utc=True)
elif np.issubdtype(np.asarray(raw_time).dtype, np.number):
    if units and " since " in units.lower():
        # Pakai metadata waktu resmi dari NetCDF (contoh: days since 2002-01-01).
        decoded = xr.coding.times.decode_cf_datetime(
            raw_time, units=units, calendar=calendar, use_cftime=None
        )
        dates = pd.to_datetime(decoded, errors="coerce", utc=True)
    else:
        # File GRACE tertentu menyimpan hitungan hari sejak 2002-01-01,
        # tetapi metadata units-nya tidak ikut tersimpan.
        # Ubah nilai ini jika metadata file Anda memakai origin berbeda.
        TIME_ORIGIN_IF_MISSING = "2002-01-01"
        TIME_UNIT_IF_MISSING = "D"
        print(
            f"PERINGATAN: koordinat {time_name!r} berupa angka tanpa metadata units. "
            f"Untuk sementara memakai {TIME_UNIT_IF_MISSING} sejak {TIME_ORIGIN_IF_MISSING}."
        )
        dates = pd.to_datetime(
            raw_time, unit=TIME_UNIT_IF_MISSING,
            origin=pd.Timestamp(TIME_ORIGIN_IF_MISSING), errors="coerce", utc=True
        )
else:
    dates = pd.to_datetime(raw_time, errors="coerce", utc=True)

if pd.isna(dates).any():
    ds.close()
    raise ValueError(
        "Ada tanggal yang tidak valid. Periksa koordinat waktu dan metadata units "
        "pada file NetCDF sebelum membuat dashboard."
    )

date_strings = [d.strftime("%Y-%m-%dT%H:%M:%S") for d in dates]
grid = np.asarray(da.values, dtype=np.float32)
grid[~np.isfinite(grid)] = np.nan
nt, ny, nx = grid.shape

valid = grid[np.isfinite(grid)]
if valid.size == 0:
    ds.close()
    raise ValueError("Semua data bernilai NaN atau tidak valid.")

vmin, vmax = float(np.nanpercentile(valid, 2)), float(np.nanpercentile(valid, 98))
if vmin == vmax:
    vmin, vmax = float(np.nanmin(valid)), float(np.nanmax(valid))
if vmin == vmax:
    vmax = vmin + 1e-6

grid_b64 = base64.b64encode(grid.astype("<f4", copy=False).tobytes()).decode("ascii")
payload = {
    "dates": date_strings, "lat": lat.tolist(), "lon": lon.tolist(),
    "nt": nt, "ny": ny, "nx": nx, "vmin": vmin, "vmax": vmax,
    "variable": var_name, "units": str(da.attrs.get("units", "")),
    "grid": grid_b64
}
payload_json = json.dumps(payload, separators=(",", ":"))

html = r'''<!doctype html>
<html lang="id"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>GRACE Temporal Explorer</title>
<style>
:root{--bg:#07111f;--panel:#10233a;--line:#243b56;--text:#eaf3ff;--muted:#91a8c3;--cyan:#4bd8e8}
*{box-sizing:border-box}body{margin:0;background:radial-gradient(ellipse at 15% 0%,#122e4a,#07111f 60%);color:var(--text);font:14px/1.5 Segoe UI,Arial,sans-serif}
.wrap{max-width:1450px;margin:auto;padding:24px}.head{display:flex;justify-content:space-between;gap:16px;align-items:flex-start;flex-wrap:wrap}
.eyebrow{color:var(--cyan);font-size:11px;letter-spacing:2px;font-weight:800}.title{font-size:clamp(25px,3vw,38px);margin:7px 0}.sub,.hint{color:var(--muted)}.sub{max-width:850px}.pill{padding:8px 12px;border:1px solid #28506a;background:#0d263b;border-radius:99px;color:#bceefa;font-size:12px}
.kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px;margin:16px 0}.card{min-width:0;padding:17px;background:linear-gradient(145deg,#10233a,#091829);border:1px solid var(--line);border-radius:17px;box-shadow:0 12px 28px #0002}
.klabel{font-size:12px;color:var(--muted)}.kvalue{font-size:clamp(20px,2.5vw,30px);font-weight:750;margin:6px 0}.layout{display:grid;grid-template-columns:minmax(0,1.65fr) minmax(290px,1fr);gap:12px}.ctitle{font-size:16px;margin:0 0 4px}.mapbox{height:450px;margin-top:13px;position:relative;background:#061321;border:1px solid #223a53;border-radius:13px;overflow:hidden}
canvas{display:block;width:100%;height:100%;image-rendering:auto}.legend{display:flex;align-items:center;gap:8px;margin-top:10px;font-size:11px;color:var(--muted)}.bar{height:9px;flex:1;border-radius:99px;background:linear-gradient(90deg,#313695,#4575b4,#74add1,#abd9e9,#ffffbf,#fdae61,#f46d43,#a50026)}
.controls{display:flex;align-items:center;gap:8px;flex-wrap:wrap;margin-top:12px}button,input,select{font:inherit;color:var(--text);background:#102941;border:1px solid #31506b;border-radius:9px;padding:9px 11px}button{cursor:pointer;font-weight:650}button:hover{border-color:var(--cyan)}input[type=range]{flex:1;min-width:120px;padding:0;accent-color:var(--cyan)}input[type=date]{color-scheme:dark}
.big{font-size:26px;font-weight:750;margin:10px 0}.mini{display:grid;grid-template-columns:1fr 1fr;gap:9px;margin-top:15px}.mini>div{background:#091a2b;border:1px solid #203b55;padding:12px;border-radius:12px}.mini b{display:block;font-size:20px;margin-top:4px}.chartbox{height:190px;margin-top:14px}
.note{font-size:12px;color:#f8d8ad;background:#2a2118;border:1px solid #6a4f2b;border-radius:12px;padding:12px;margin-top:12px}.tablewrap{max-height:220px;overflow:auto;margin-top:12px}table{width:100%;border-collapse:collapse}td,th{text-align:left;padding:8px;border-bottom:1px solid #20364d;font-variant-numeric:tabular-nums}th{position:sticky;top:0;background:#10233a;color:var(--muted)}td:last-child,th:last-child{text-align:right}footer{padding:20px 2px;color:#718aa6;font-size:11px}
@media(max-width:900px){.layout{grid-template-columns:1fr}.kpis{grid-template-columns:repeat(2,1fr)}.mapbox{height:370px}}@media(max-width:480px){.wrap{padding:12px}.card{padding:13px}.mapbox{height:300px}}
</style></head><body><div class="wrap">
<header class="head"><div><div class="eyebrow">EARTH OBSERVATION · GRACE / GRACE-FO</div><h1 class="title">GRACE Temporal Explorer</h1><div class="sub">Peta grid mengikuti waktu, dengan animasi, kalender, statistik temporal, dan deret waktu sel grid.</div></div><div class="pill" id="period">Memuat periode…</div></header>
<section class="card" style="margin-top:14px;margin-bottom:14px">
<h2 class="ctitle">Tentang data GRACE</h2>
<div class="sub" style="margin-top:7px">
Peta menampilkan distribusi spasial variabel <b id="aboutVar">GRACE</b> pada setiap waktu pengamatan.
Data GRACE/GRACE-FO umumnya digunakan untuk mengamati perubahan penyimpanan air daratan melalui variasi medan gravitasi.
Interpretasi satuan dan makna tepat nilai mengikuti metadata variabel pada file NetCDF; jika variabelnya <b>lwe_thickness</b>, nilainya biasanya merepresentasikan ketebalan ekuivalen air cair (liquid water equivalent thickness).
Gunakan rentang waktu dan satuan pada metadata sumber saat membuat interpretasi ilmiah.
</div></section>
<section class="kpis">
<div class="card"><div class="klabel">Observasi waktu</div><div class="kvalue" id="nobs">—</div><div class="hint">frame temporal</div></div>
<div class="card"><div class="klabel">Rata-rata frame</div><div class="kvalue" id="mean">—</div><div class="hint" id="units">Unit dari NetCDF</div></div>
<div class="card"><div class="klabel">Nilai minimum</div><div class="kvalue" id="min">—</div><div class="hint">grid aktif</div></div>
<div class="card"><div class="klabel">Nilai maksimum</div><div class="kvalue" id="max">—</div><div class="hint">grid aktif</div></div>
</section>
<section class="layout"><div class="card"><h2 class="ctitle">Peta temporal global</h2><div class="hint" id="mapHint">Pilih tanggal atau jalankan animasi.</div>
<div class="mapbox"><canvas id="map"></canvas></div><div class="legend"><span id="vmin">min</span><div class="bar"></div><span id="vmax">max</span></div>
<div class="controls"><button id="prev">◀ Sebelumnya</button><button id="play">▶ Putar</button><button id="next">Berikutnya ▶</button><input id="slider" type="range" min="0" max="0" value="0"><select id="speed"><option value="1200">Lambat</option><option value="650" selected>Normal</option><option value="250">Cepat</option></select></div>
<div class="controls"><label class="klabel" for="date">Pilih tanggal</label><input type="date" id="date"><span class="hint" id="frame">Frame —</span></div>
 </div>
<div class="card"><h2 class="ctitle">Detail frame aktif</h2><div class="hint">Statistik seluruh sel valid pada tanggal terpilih</div><div class="big" id="activeDate">—</div><div class="big" id="activeMean">—</div><div class="hint" id="activeUnit">Rata-rata grid</div>
<div class="mini"><div><div class="klabel">Median</div><b id="median">—</b></div><div><div class="klabel">Simpangan baku</div><b id="std">—</b></div><div><div class="klabel">Sel valid</div><b id="valid">—</b></div><div><div class="klabel">Variabel</div><b id="varname" style="font-size:14px;overflow-wrap:anywhere">—</b></div></div>
<h2 class="ctitle" style="margin-top:20px">Deret waktu sel terpilih</h2><div class="hint">Klik lokasi pada peta untuk memilih sel.</div><div class="chartbox"><canvas id="series"></canvas></div><div class="hint" id="cellInfo">Belum ada sel dipilih.</div></div></section>
<section class="card" style="margin-top:12px"><h2 class="ctitle">Sampel nilai grid pada tanggal aktif</h2><div class="hint">Klik baris untuk memilih lokasi.</div><div class="tablewrap"><table><thead><tr><th>Lintang</th><th>Bujur</th><th>Nilai</th></tr></thead><tbody id="cells"></tbody></table></div></section>
<footer>HTML mandiri · data tertanam dalam file · tanpa Plotly atau koneksi internet.</footer></div>
<script>
const D=__PAYLOAD__;
const bytes=Uint8Array.from(atob(D.grid),c=>c.charCodeAt(0));
const G=new Float32Array(bytes.buffer,bytes.byteOffset,bytes.byteLength/4);
const dates=D.dates.map(s=>new Date(s+'Z')),N=D.nt,NY=D.ny,NX=D.nx,total=NY*NX;
const fmt=n=>Number(n).toLocaleString('id-ID',{maximumFractionDigits:4,minimumFractionDigits:2});
const dateFmt=d=>d.toLocaleDateString('id-ID',{day:'2-digit',month:'short',year:'numeric',timeZone:'UTC'});
document.getElementById('nobs').textContent=N;document.getElementById('period').textContent=dateFmt(dates[0])+' — '+dateFmt(dates[N-1]);
document.getElementById('units').textContent=D.units||'Unit tidak tercantum';document.getElementById('activeUnit').textContent=D.units||'Unit mengikuti NetCDF';document.getElementById('varname').textContent=D.variable;document.getElementById('aboutVar').textContent=D.variable;
document.getElementById('vmin').textContent=fmt(D.vmin);document.getElementById('vmax').textContent=fmt(D.vmax);
const slider=document.getElementById('slider'),dateInput=document.getElementById('date'),map=document.getElementById('map'),mc=map.getContext('2d'),series=document.getElementById('series'),sc=series.getContext('2d');
slider.max=N-1;dateInput.min=D.dates[0].slice(0,10);dateInput.max=D.dates[N-1].slice(0,10);
let idx=0,playing=false,timer=null,selected={y:Math.floor(NY/2),x:Math.floor(NX/2)},mapCells=[];
function resizeCanvas(c,ctx){const r=c.getBoundingClientRect(),d=window.devicePixelRatio||1;c.width=Math.round(r.width*d);c.height=Math.round(r.height*d);ctx.setTransform(d,0,0,d,0,0)}
function valueAt(t,y,x){return G[t*total+y*NX+x]}
function color(v){let t=Math.max(0,Math.min(1,(v-D.vmin)/(D.vmax-D.vmin))),s=[[49,54,149],[69,117,180],[116,173,209],[171,217,233],[255,255,191],[253,174,97],[244,109,67],[165,0,38]],z=t*(s.length-1),a=Math.floor(z),f=z-a,b=s[Math.min(a+1,s.length-1)],c=s[a];return `rgb(${Math.round(c[0]+(b[0]-c[0])*f)},${Math.round(c[1]+(b[1]-c[1])*f)},${Math.round(c[2]+(b[2]-c[2])*f)})`}
function stats(t){let a=[];for(let i=0;i<total;i++){let v=G[t*total+i];if(Number.isFinite(v))a.push(v)}if(!a.length)return null;a.sort((a,b)=>a-b);let sum=0;for(let v of a)sum+=v;let m=sum/a.length,ss=0;for(let v of a)ss+=(v-m)**2;return {n:a.length,mean:m,median:a.length%2?a[(a.length-1)/2]:(a[a.length/2-1]+a[a.length/2])/2,std:Math.sqrt(ss/a.length),min:a[0],max:a[a.length-1]}}
function drawMap(){let w=map.clientWidth,h=map.clientHeight;mc.clearRect(0,0,w,h);mc.fillStyle='#061321';mc.fillRect(0,0,w,h);let p={l:42,r:12,t:14,b:30},pw=w-p.l-p.r,ph=h-p.t-p.b;mapCells=[];
let lonMin=Math.min(...D.lon),lonMax=Math.max(...D.lon),latMin=Math.min(...D.lat),latMax=Math.max(...D.lat);if(lonMin===lonMax){lonMin-=.5;lonMax+=.5}if(latMin===latMax){latMin-=.5;latMax+=.5}
const xp=v=>p.l+(v-lonMin)/(lonMax-lonMin)*pw,yp=v=>p.t+(latMax-v)/(latMax-latMin)*ph;
mc.strokeStyle='rgba(145,168,195,.18)';mc.fillStyle='#91a8c3';mc.font='10px Segoe UI,Arial';mc.lineWidth=1;
for(let k=0;k<=4;k++){let xx=p.l+pw*k/4,yy=p.t+ph*k/4;mc.beginPath();mc.moveTo(xx,p.t);mc.lineTo(xx,h-p.b);mc.stroke();mc.beginPath();mc.moveTo(p.l,yy);mc.lineTo(w-p.r,yy);mc.stroke();mc.textAlign='center';mc.fillText((lonMin+(lonMax-lonMin)*k/4).toFixed(0)+'°',xx,h-10);mc.textAlign='right';mc.fillText((latMax-(latMax-latMin)*k/4).toFixed(0)+'°',p.l-7,yy+3)}
let cw=pw/NX+0.5,ch=ph/NY+0.5;for(let y=0;y<NY;y++)for(let x=0;x<NX;x++){let v=valueAt(idx,y,x);if(!Number.isFinite(v))continue;let xx=xp(D.lon[x]),yy=yp(D.lat[y]);mc.fillStyle=color(v);mc.fillRect(xx-cw/2,yy-ch/2,cw,ch);mapCells.push({x,y,px:xx,py:yy})}
let sx=xp(D.lon[selected.x]),sy=yp(D.lat[selected.y]);mc.strokeStyle='#fff';mc.lineWidth=2;mc.beginPath();mc.arc(sx,sy,6,0,Math.PI*2);mc.stroke();mc.strokeStyle='#ffb86b';mc.beginPath();mc.arc(sx,sy,8,0,Math.PI*2);mc.stroke()}
function drawSeries(){let w=series.clientWidth,h=series.clientHeight;sc.clearRect(0,0,w,h);let p={l:44,r:10,t:12,b:28},pw=w-p.l-p.r,ph=h-p.t-p.b,a=[];for(let t=0;t<N;t++){let v=valueAt(t,selected.y,selected.x);if(Number.isFinite(v))a.push({t,v})}if(!a.length)return;
let lo=Math.min(...a.map(q=>q.v)),hi=Math.max(...a.map(q=>q.v));if(lo===hi){lo-=1;hi+=1}const xp=t=>p.l+t/Math.max(1,N-1)*pw,yp=v=>p.t+(hi-v)/(hi-lo)*ph;
sc.strokeStyle='rgba(145,168,195,.18)';sc.fillStyle='#91a8c3';sc.font='10px Segoe UI,Arial';for(let k=0;k<=3;k++){let yy=p.t+ph*k/3;sc.beginPath();sc.moveTo(p.l,yy);sc.lineTo(w-p.r,yy);sc.stroke();sc.textAlign='right';sc.fillText(fmt(hi-(hi-lo)*k/3),p.l-6,yy+3)}
sc.strokeStyle='#4bd8e8';sc.lineWidth=1.7;sc.beginPath();a.forEach((q,j)=>j?sc.lineTo(xp(q.t),yp(q.v)):sc.moveTo(xp(q.t),yp(q.v)));sc.stroke();let v=valueAt(idx,selected.y,selected.x);if(Number.isFinite(v)){sc.fillStyle='#ffb86b';sc.beginPath();sc.arc(xp(idx),yp(v),4,0,Math.PI*2);sc.fill()}
sc.textAlign='center';sc.fillStyle='#91a8c3';sc.fillText(dates[0].getUTCFullYear(),p.l,h-8);sc.fillText(dates[Math.floor((N-1)/2)].getUTCFullYear(),p.l+pw/2,h-8);sc.fillText(dates[N-1].getUTCFullYear(),w-p.r,h-8)}
function buildCells(){let tb=document.getElementById('cells');tb.innerHTML='';let stride=Math.max(1,Math.ceil(total/120));for(let i=0;i<total;i+=stride){let y=Math.floor(i/NX),x=i%NX,v=valueAt(idx,y,x);if(!Number.isFinite(v))continue;let tr=document.createElement('tr');tr.innerHTML=`<td>${fmt(D.lat[y])}°</td><td>${fmt(D.lon[x])}°</td><td>${fmt(v)}</td>`;tr.style.cursor='pointer';tr.onclick=()=>{selected={y,x};update()};tb.appendChild(tr)}}
function update(){slider.value=idx;dateInput.value=D.dates[idx].slice(0,10);document.getElementById('frame').textContent=`Frame ${idx+1} / ${N}`;document.getElementById('activeDate').textContent=dateFmt(dates[idx]);document.getElementById('mapHint').textContent=dateFmt(dates[idx])+' · '+D.variable;let s=stats(idx);
for(let id of ['mean','min','max','activeMean','median','std','valid'])document.getElementById(id).textContent='Tidak tersedia';
if(s){document.getElementById('mean').textContent=fmt(s.mean);document.getElementById('min').textContent=fmt(s.min);document.getElementById('max').textContent=fmt(s.max);document.getElementById('activeMean').textContent=fmt(s.mean);document.getElementById('median').textContent=fmt(s.median);document.getElementById('std').textContent=fmt(s.std);document.getElementById('valid').textContent=s.n.toLocaleString('id-ID')}
let cv=valueAt(idx,selected.y,selected.x);document.getElementById('cellInfo').textContent=`Lintang ${fmt(D.lat[selected.y])}°, bujur ${fmt(D.lon[selected.x])}° · nilai ${Number.isFinite(cv)?fmt(cv):'tidak tersedia'}`;drawMap();drawSeries();buildCells()}
function stop(){playing=false;clearInterval(timer);timer=null;document.getElementById('play').textContent='▶ Putar'}
document.getElementById('prev').onclick=()=>{stop();idx=Math.max(0,idx-1);update()};document.getElementById('next').onclick=()=>{stop();idx=Math.min(N-1,idx+1);update()};slider.oninput=()=>{idx=+slider.value;update()};
dateInput.onchange=()=>{let target=new Date(dateInput.value+'T00:00:00Z').getTime(),best=0,dist=Infinity;dates.forEach((d,i)=>{let z=Math.abs(d.getTime()-target);if(z<dist){dist=z;best=i}});idx=best;update()};
document.getElementById('play').onclick=()=>{if(playing){stop();return}playing=true;document.getElementById('play').textContent='Ⅱ Jeda';timer=setInterval(()=>{if(idx>=N-1){stop();return}idx++;update()},+document.getElementById('speed').value)};
document.getElementById('speed').onchange=()=>{if(playing){stop();document.getElementById('play').click()}};
map.addEventListener('click',e=>{let r=map.getBoundingClientRect(),mx=e.clientX-r.left,my=e.clientY-r.top,best=null,dist=Infinity;for(let q of mapCells){let d=(q.px-mx)**2+(q.py-my)**2;if(d<dist){dist=d;best=q}}if(best){selected={y:best.y,x:best.x};update()}});
function resize(){resizeCanvas(map,mc);resizeCanvas(series,sc);update()}window.addEventListener('resize',resize);resize();
</script></body></html>'''

html = html.replace("__PAYLOAD__", payload_json)
out_path = Path(OUTPUT_HTML)
out_path.write_text(html, encoding="utf-8")
ds.close()

print(f"Dashboard berhasil dibuat: {out_path.resolve()}")
print(f"Variabel: {var_name}; grid setelah downsample: time={nt}, lat={ny}, lon={nx}")
print(f"Periode: {date_strings[0]} sampai {date_strings[-1]}")
print(f"Rentang warna tetap: {vmin:.4f} sampai {vmax:.4f}")
webbrowser.open(out_path.resolve().as_uri())


Variabel tersedia: ['time_bounds', 'lwe_thickness']
Koordinat tersedia: ['time', 'lon', 'lat']
<xarray.Dataset> Size: 1GB
Dimensions:        (time: 260, timebound: 2, lat: 720, lon: 1440)
Coordinates:
  * time           (time) float32 1kB 107.0 129.5 227.5 ... 8.962e+03 8.994e+03
  * lat            (lat) float32 3kB -89.88 -89.62 -89.38 ... 89.38 89.62 89.88
  * lon            (lon) float32 6kB 0.125 0.375 0.625 ... 359.4 359.6 359.9
Dimensions without coordinates: timebound
Data variables:
    time_bounds    (time, timebound) float32 2kB ...
    lwe_thickness  (time, lat, lon) float32 1GB ...
Attributes: (12/59)
    Conventions:                CF-1.6, ACDD-1.3, ISO 8601
    filename:                   netcdf/CSR_GRACE_GRACE-FO_RL0603_Mascons_all-...
    Metadata_Conventions:       Unidata Dataset Discovery v1.0
    standard_name_vocabulary:   NetCDF Climate and Forecast (CF) Metadata Con...
    title:                      CSR GRACE and GRACE-FO MASCON RL0603M
    subtitle:            

True